# 选修E3 · Day 2 上机：LLM 应用工程

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

**核心命题**：用 RAG 让 LLM 基于产品知识库生成营销文案，Prompt 工程控制输出，LangSmith 追踪全链路，RAGAS 评估质量。

**真实库**：tiktoken（token 计数）+ langchain_core（Prompt 模板）+ langsmith（追踪）+ numpy（RAG 检索）


## 0. 环境准备

首次运行需安装依赖（取消注释执行一次）：

> ⚠️ 全部使用本地库，无需 OPENAI_API_KEY。无 API key 时用 mock LLM + Prompt 模板演示。
> tiktoken / langchain-core / langsmith / numpy 均为纯本地库，秒级加载。


In [ ]:
# !pip install tiktoken langchain-core langsmith numpy -q

import tiktoken
import numpy as np
import math
import json
import time
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langsmith import traceable

print("=== 环境就绪 ===")
print(f"tiktoken: {tiktoken.__version__}")
print(f"numpy: {np.__version__}")
print(f"langchain_core: 可用")
print(f"langsmith: 可用")


## 1. 场景背景与营销映射

**核心命题**：LLM 应用工程是营销 Agent 的"应用层"。本上机解决一个完整场景：
- 用 **tiktoken** 计算营销文案 token 数 + 推理成本（gpt-4o vs DeepSeek V3）
- 用 **ChatPromptTemplate** 构建营销文案生成 Prompt
- 用 **@traceable** 追踪 LLM 调用
- 用 **numpy TF-IDF** 实现 RAG 检索（营销知识库）
- 用 **RAG + Prompt + mock LLM** 生成基于知识库的营销文案
- 用 **RAGAS 简化实现** 评估 RAG 质量

**营销知识库**（智能手表产品文档，5 个文档）：


In [ ]:
# 营销知识库：智能手表产品文档（基于真实产品文档结构构建，见 data/README.md）
knowledge_base = [
    {"id": "doc1", "content": "智能手表Pro支持7天超长续航，采用低功耗芯片2.0，日常使用可达7天，运动模式3天。"},
    {"id": "doc2", "content": "手表内置100+运动模式，包括跑步、游泳、骑行、瑜伽，支持自动识别6种运动。"},
    {"id": "doc3", "content": "心率血氧监测功能，24小时连续心率监测，血氧饱和度SpO2测量，异常心率提醒。"},
    {"id": "doc4", "content": "手表防水等级5ATM，支持游泳佩戴，50米防水深度。"},
    {"id": "doc5", "content": "品牌故事：致力于用科技守护健康，让每个人都能享受智能穿戴带来的便利。"},
]

print(f"知识库文档数: {len(knowledge_base)}")
for doc in knowledge_base:
    print(f"  {doc['id']}: {doc['content'][:40]}...")


## TODO 1：用 tiktoken 计算营销文案 token 数 + 推理成本

**tiktoken** 是 OpenAI 的 BPE 分词器，精确计算 token 数：
- `o200k_base`：gpt-4o 的编码
- `cl100k_base`：gpt-4/3.5 和 DeepSeek V3 的编码

**任务**：
1. 用 `tiktoken.get_encoding('o200k_base')` 和 `cl100k_base` 加载编码器
2. 对中英文营销文案做 tokenization，对比 token 消耗
3. 结合 gpt-4o（$2.50/$10.00 per M）和 DeepSeek V3（$0.27/$1.10 per M）定价，计算日均 1000 次调用的月成本


In [ ]:
# ===== 你的代码 =====
# TODO: 你的代码
# 1) 用 tiktoken.get_encoding('o200k_base') 和 'cl100k_base' 加载两个编码器
# 2) 对 marketing_copy_zh 和 marketing_copy_en 做 tokenization，打印 token 数
# 3) 结合 gpt-4o 和 DeepSeek V3 定价，计算日均 1000 次调用（input 500 tokens, output 200 tokens）的月成本
raise NotImplementedError


## TODO 2：用 ChatPromptTemplate 构建营销文案生成 Prompt

**ChatPromptTemplate**（langchain_core）构建结构化 Prompt：
- `from_messages([("system", ...), ("human", ...)])` 定义 System + Human 消息
- `format_messages(...)` 填充变量生成消息列表

**任务**：
1. 用 ChatPromptTemplate.from_messages 构建 Prompt（system 定义角色+约束，human 定义产品信息）
2. 用 format_messages 填充 product_name / selling_points / target_audience
3. 打印格式化后的消息列表


In [ ]:
# ===== 你的代码 =====
# TODO: 你的代码
# 1) 用 ChatPromptTemplate.from_messages 构建 Prompt
#    - system: "你是一个营销文案专家。请根据产品信息生成吸引人的营销文案。要求：1) 突出核心卖点 2) 包含CTA 3) 控制在100字以内"
#    - human: "产品名称：{product_name}\n核心卖点：{selling_points}\n目标受众：{target_audience}\n请生成营销文案。"
# 2) 用 format_messages 填充：product_name="智能手表Pro", selling_points="7天续航, 100+运动模式, 心率血氧监测", target_audience="25-35岁都市白领"
# 3) 打印每条消息的 type 和 content
raise NotImplementedError


## TODO 3：用 @traceable 追踪 LLM 调用（mock LLM）

**langsmith @traceable** 装饰器记录函数调用全链路。无 LANGSMITH_API_KEY 时仍可运行（本地模式）。

**任务**：
1. 定义 `@traceable(name="marketing_copy_generator")` 装饰的函数
2. 函数内部用 ChatPromptTemplate 构建 Prompt，用 mock LLM（模板生成）输出营销文案
3. 调用函数并打印结果


In [ ]:
# ===== 你的代码 =====
# TODO: 你的代码
# 1) 用 @traceable(name="marketing_copy_generator") 装饰一个函数 generate_marketing_copy(product_name, selling_points, target_audience)
# 2) 函数内部用 ChatPromptTemplate 构建 Prompt 并 format_messages
# 3) 用 mock LLM（模板生成）输出文案: f"【{product_name}】{selling_points}，专为{target_audience}打造。立即抢购，享受健康生活！"
# 4) 返回 {"copy": copy, "prompt_messages": len(messages), "tokens_estimated": len(copy)}
# 5) 调用函数并打印结果
raise NotImplementedError


## TODO 4：用 numpy TF-IDF + 余弦相似度实现 RAG 检索

**RAG 检索**：从营销知识库中召回与用户问题最相关的文档。
- **TF-IDF**：词频-逆文档频率，将文档转为向量
- **余弦相似度**：衡量 query 与 doc 的相似度

**任务**：
1. 实现 `tfidf_vectorize(docs, query)` 函数：构建词表 -> 计算 TF-IDF -> 余弦相似度
2. 对 query="手表续航多久" 检索，打印 top-3 召回结果
3. （可选）对比 sentence-transformers all-MiniLM-L6-v2 的检索效果


In [ ]:
# ===== 你的代码 =====
# TODO: 你的代码
# 1) 实现 tfidf_vectorize(docs, query) 函数:
#    - 构建词表（docs + query 中所有字符的集合）
#    - 计算 TF（词频/文档长度）
#    - 计算 IDF（log((N+1)/(df+1)) + 1）
#    - doc_vectors = TF * IDF, query_vector = TF * IDF
#    - 余弦相似度 = dot(a,b) / (||a|| * ||b||)
# 2) 对 query="手表续航多久" 检索 knowledge_base，打印 top-3 召回
raise NotImplementedError


## TODO 5：用 RAG + Prompt + mock LLM 生成基于知识库的营销文案

**RAG 生成**：检索 -> Prompt 模板 -> LLM 生成

**任务**：
1. 定义 `@traceable(name="rag_marketing_qa")` 装饰的函数
2. 函数内部：检索 top-3 -> ChatPromptTemplate 构建 RAG Prompt -> mock LLM 生成回答
3. 调用函数问"智能手表的续航时间是多少？"，打印回答 + 检索来源


In [ ]:
# ===== 你的代码 =====
# TODO: 你的代码
# 1) 定义 @traceable(name="rag_marketing_qa") 装饰的函数 rag_marketing_qa(question, knowledge_base, top_k=3)
# 2) 函数内部:
#    - 调用 tfidf_vectorize 检索 top_k 文档
#    - 用 ChatPromptTemplate 构建 RAG Prompt（system: 基于上下文回答，不编造；human: 上下文+问题）
#    - mock LLM 生成: f"根据产品资料：{top_doc_content[:80]}"
# 3) 返回 {"question": question, "answer": answer, "retrieved_docs": [...], "context_used": context[:200]}
# 4) 调用函数问"智能手表的续航时间是多少？"，打印回答 + 检索来源
raise NotImplementedError


## TODO 6：用 RAGAS 简化实现评估 RAG 质量

**RAGAS**（Retrieval-Augmented Generation Assessment）核心指标：
- **Faithfulness（忠实度）**：回答信息是否能在检索上下文中找到（防幻觉）
- **Context Recall（上下文召回率）**：ground truth 信息是否被检索到
- **Answer Relevance（回答相关性）**：回答是否切题

**任务**：
1. 实现 `ragas_simplified(question, answer, retrieved_docs, ground_truth)` 函数
2. 用字符重叠规则近似 faithfulness / context_recall / answer_relevance
3. 评估 TODO5 的 RAG 结果


In [ ]:
# ===== 你的代码 =====
# TODO: 你的代码
# 1) 实现 ragas_simplified(question, answer, retrieved_docs, ground_truth=None):
#    - faithfulness: answer 字符集与 context 字符集的交集比例
#    - context_recall: ground_truth 字符集与 context 字符集的交集比例（无 ground_truth 时为 1.0）
#    - answer_relevance: question 字符集与 answer 字符集的交集比例
# 2) 评估 TODO5 的 result（question/answer/retrieved_docs）+ ground_truth="智能手表Pro支持7天超长续航"
# 3) 打印三个指标
raise NotImplementedError


## 总结

**本上机完成了 LLM 应用工程的全链路**：

| TODO | 任务 | 真实库 | 营销场景 |
|------|------|--------|---------|
| TODO1 | Token 计数 + 推理成本 | tiktoken | gpt-4o vs DeepSeek V3 定价对比 |
| TODO2 | Prompt 模板 | langchain_core ChatPromptTemplate | 营销文案生成 Prompt |
| TODO3 | LLM 追踪 | langsmith @traceable | 营销文案生成全链路追踪 |
| TODO4 | RAG 检索 | numpy TF-IDF + 余弦相似度 | 营销知识库召回 |
| TODO5 | RAG 生成 | langchain_core + langsmith | 基于知识库的营销文案 |
| TODO6 | RAGAS 评估 | numpy 规则实现 | RAG 质量评估 |

**关键收获**：
- DeepSeek V3 推理成本仅为 gpt-4o 的 ~10%，MoE 架构的革命性意义
- Prompt Engineering 是 LLM 应用的第一道工具（成本极低）
- RAG 解决 LLM 知识静态 + 无法访问私有数据的问题
- LangSmith @traceable 是 LLM 应用可观测性的标配
- RAGAS 评估是 RAG 系统持续优化的基础

**下一步**：Day 3 LLM 评估与部署--从 RAGAS 扩展到 MMLU/LLM-as-Judge 完整评估体系。
